<a href="https://colab.research.google.com/github/ShinAsakawa/ShinAsakawa.github.io/blob/master/2026notebooks/2026_0422WLSP_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import Levenshtein
except ImportError:
    !pip install Levenshtein
    import Levenshtein

try:
    import jamorasep
except ImportError:
    !pip install jamorasep
    import jamorasep

In [ ]:
"""
WLSP_metrics.py — 心理言語学変数の一括計算
================================================

入力: (単語, 読み) のペア
出力: ONS, PNS, OLD20

全て WLSP の規則・母集団に基づいて計算
読みは WLSP 方式（長音→母音化, カタカナ→ひらがな）に正規化

ONS, PNS は重複排除（unique）でカウントする（Coltheart's N の本来の定義）。
JALEX は重複許容（with_duplicates）でカウントしている可能性があるため、
JALEX の公式値と突き合わせる場合は注意が必要。

依存: pandas

使い方:
    engine = WLSP_metrics_Engine(wlsp_path)
    result = engine.compute("研究", "ケンキュウ")
    print(result)
"""

import math
import unicodedata
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from itertools import combinations, product
from typing import List, Tuple, Dict, Optional, NamedTuple
from Levenshtein import distance as _levenshtein

# ╔═══════════════════════════════════════════════════════════╗
# ║  1. WLSP 読み正規化（長音規則）                            ║
# ╚═══════════════════════════════════════════════════════════╝

_VOWEL_KATA = {}
_ROWS = {
    'ア': 'アカサタナハマヤラワガザダバパァャヮ',
    'イ': 'イキシチニヒミリギジヂビピィ',
    'ウ': 'ウクスツヌフムユルグズヅブプゥュヴ',
    'エ': 'エケセテネヘメレゲゼデベペェ',
    'オ': 'オコソトノホモヨロヲゴゾドボポォョ',
}
for _v, _cs in _ROWS.items():
    for _c in _cs:
        _VOWEL_KATA[_c] = _v

_VOWEL_HIRA = {}
_V2H = {'ア': 'あ', 'イ': 'い', 'ウ': 'う', 'エ': 'え', 'オ': 'お'}
for _c, _v in _VOWEL_KATA.items():
    _cp = ord(_c)
    if 0x30A1 <= _cp <= 0x30F6:
        _VOWEL_HIRA[chr(_cp - 0x60)] = _V2H[_v]
    elif _c == 'ヴ':
        _VOWEL_HIRA['ゔ'] = 'う'
    elif _c == 'ヮ':
        _VOWEL_HIRA[chr(ord('ヮ') - 0x60)] = _V2H[_v]

VOWEL_MAP = {**_VOWEL_KATA, **_VOWEL_HIRA}
SPECIAL_MORA = frozenset('ンッーんっ')


def resolve_chouon(text: str) -> str:
    """ー を直前仮名の母音段に解決する（WLSP 規則 R1-R4）。"""
    if not isinstance(text, str):
        return text
    chars = list(text)
    for i, ch in enumerate(chars):
        if ch != 'ー':
            continue
        for j in range(i - 1, -1, -1):
            v = VOWEL_MAP.get(chars[j])
            if v is not None:
                chars[i] = v
                break
            if chars[j] not in SPECIAL_MORA:
                break
    return ''.join(chars)


def kata_to_hira(text: str) -> str:
    """カタカナ → ひらがな変換（R3）。"""
    if not isinstance(text, str):
        return text
    result = []
    for ch in text:
        cp = ord(ch)
        if 0x30A1 <= cp <= 0x30F6:
            result.append(chr(cp - 0x60))
        elif ch == 'ヴ':
            result.append('ゔ')
        else:
            result.append(ch)
    return ''.join(result)


def normalize_reading(reading: str) -> str:
    """読みを WLSP 方式に正規化する。"""
    if not isinstance(reading, str):
        return reading
    reading = resolve_chouon(reading)
    reading = kata_to_hira(reading)
    return reading


def primary_yomi(y: str) -> str:
    """・含む読みから代表形（先頭）を取る。"""
    return y.split('・')[0] if '・' in y else y


# ╔═══════════════════════════════════════════════════════════╗
# ║  2. モーラ分解                                            ║
# ╚═══════════════════════════════════════════════════════════╝

_COMBO_HIRA = set('ゃゅょぁぃぅぇぉゎ')
_SPECIAL_HIRA = set('んっ')


def tokenize_mora(reading: str) -> tuple:
    """ひらがな読みをモーラ単位に分解し tuple で返す。"""
    if not isinstance(reading, str) or reading == '':
        return ()
    morae = jamorasep.parse(reading, output_format="katakana")
    return tuple(morae)

# ╔═══════════════════════════════════════════════════════════╗
# ║  3. ONS / PNS（テンプレートインデックス）                     ║
# ║     重複排除（unique）: 同一語形は1回だけカウント              ║
# ╚═══════════════════════════════════════════════════════════╝

SENTINEL = '\x00'


def _build_ons_index(words):
    """正書法テンプレートインデックス（重複排除: set）。"""
    idx = defaultdict(set)
    for w in words:
        for i in range(len(w)):
            idx[w[:i] + SENTINEL + w[i + 1:]].add(w)
    return idx


def _compute_ons(target, idx):
    nb = set()
    for i in range(len(target)):
        nb.update(idx.get(target[:i] + SENTINEL + target[i + 1:], set()))
    nb.discard(target)
    return len(nb)


def _build_pns_index(mora_tuples):
    """音韻テンプレートインデックス（重複排除: set）。"""
    idx = defaultdict(set)
    for mt in mora_tuples:
        for i in range(len(mt)):
            idx[mt[:i] + (SENTINEL,) + mt[i + 1:]].add(mt)
    return idx


def _compute_pns(mora_tuple, idx):
    nb = set()
    for i in range(len(mora_tuple)):
        nb.update(idx.get(mora_tuple[:i] + (SENTINEL,) + mora_tuple[i + 1:],
                          set()))
    nb.discard(mora_tuple)
    return len(nb)


# ╔═══════════════════════════════════════════════════════════╗
# ║  4. OLD20（Levenshtein距離ベース）                         ║
# ╚═══════════════════════════════════════════════════════════╝

def _compute_old20(target, lexicon_list, k=20):
    """平均 orthographic Levenshtein distance to k nearest neighbors."""
    dists = sorted(_levenshtein(target, w)
                   for w in lexicon_list if w != target)
    if len(dists) >= k:
        return sum(dists[:k]) / k
    elif dists:
        return sum(dists) / len(dists)
    else:
        return float('nan')


# ── Mora segmentation (original: artifact 1e7534f4) ──

SMALL_KANA = set(list("ゃゅょぁぃぅぇぉャュョァィゥェォ"))
LONG_VOWEL = "ー"
SOKUON = {"っ", "ッ"}
HATSUON = {"ん", "ン"}

def kana_to_mora(reading: str) -> List[str]:
    """カタカナ/ひらがな混在の読みをモーラ単位に分解する。"""
    return jamorasep.parse(reading, output_format="katakana")

# ╔═══════════════════════════════════════════════════════════╗
# ║  6. 統合エンジン                                           ║
# ╚═══════════════════════════════════════════════════════════╝

WLSP_COL_NAMES = [
    'レコードID番号', '見出し番号', 'レコード種別', '類', '部門',
    '中項目', '分類項目', '分類番号', '段落番号', '小段落番号', '語番号',
    '見出し', '見出し本体', '読み', '逆読み'
]


class WordMetrics(NamedTuple):
    """計算結果。"""
    word: str               # 入力語
    reading_normalized: str  # WLSP方式に正規化された読み
    n_chars: int            # 文字数
    n_morae: int            # モーラ数
    ons: int                # Orthographic Neighborhood Size
    pns: int                # Phonological Neighborhood Size
    old20: float            # Orthographic Levenshtein Distance 20

class WLSP_metrics_Engine:
    """WLSP ベースの心理言語学変数計算エンジン。

    初期化時に以下の前処理を自動的に行う:
      1. レコード種別 B の除外（・／併記の分割元）
      2. 区切り記号（＊）、用言パターン（…）、接辞（−）、ゲタ記号（〓）の除外
      3. NFKC 正規化（全角英数字→半角等。除外後に適用）
      4. ONS/PNS インデックス構築（重複排除 unique）

    ONS/PNS は重複排除（unique）でカウントする。
    同一語形が WLSP 内に複数レコード存在しても 1 回だけカウントする。

    Parameters
    ----------
    wlsp_path : str
        bunruidb.txt のパス。
    """

    def __init__(self, wlsp_path: str, em_n_iter: int = 20,
                 em_max_chars: int = 4):
        print("[1/5] WLSP 読み込み...")
        raw_df = pd.read_csv(
            wlsp_path, header=None, encoding='shift-jis',
            names=WLSP_COL_NAMES)
        print(f"       {len(raw_df)} レコード")

        # ── WLSP データ修正（読みの誤り）──
        raw_df = raw_df.copy()
        _YOMI_FIXES = {
            'ウォッカ': 'うぉっか',   # 原データ: うおっか
            'だけど':   'だけど',     # 原データ: たけど
        }
        for midashi, correct_yomi in _YOMI_FIXES.items():
            mask = raw_df['見出し本体'] == midashi
            if mask.any():
                raw_df.loc[mask, '読み'] = correct_yomi

        # ── レコード除外 ──
        print("[2/5] レコード除外...")
        df = raw_df.copy()
        n0 = len(df)
        # B種（・／併記の分割元）を除外
        df = df[df['レコード種別'] != 'B']
        # (1) ＊のみのレコード（分類区切り, 語番号=99）を除外
        df = df[df['見出し本体'] != '＊']
        # (2) 三点リーダから始まるレコード（用言パターン）を除外
        df = df[~df['見出し本体'].str.startswith('…', na=False)]
        # (3) 接辞（−で始まるか終わるもの）を除外。語中の−（例: ＣＤ−Ｒ）は残す
        df = df[~(df['見出し本体'].str.startswith('−', na=False) |
                  df['見出し本体'].str.endswith('−', na=False))]
        # (4) ゲタ記号（〓）を含むレコードを除外
        df = df[~df['見出し本体'].str.contains('〓', na=False)]
        print(f"       {n0} → {len(df)} レコード（{n0 - len(df)} 件除外）")

        # ── NFKC 正規化（除外後に適用）──
        print("[3/5] NFKC 正規化...")
        df = df.copy()
        df['見出し本体'] = df['見出し本体'].apply(
            lambda x: unicodedata.normalize('NFKC', x)
            if isinstance(x, str) else x)
        df['読み'] = df['読み'].apply(
            lambda x: unicodedata.normalize('NFKC', x)
            if isinstance(x, str) else x)
        self.wlsp_df = df

        # ── ONS（書記近傍サイズ）──
        print("[4/5] ONS インデックス構築...")
        # 重複排除: 同一見出し本体は1回だけカウント
        self.orth_set = set(df['見出し本体'].dropna().unique())
        self.orth_list = list(self.orth_set)
        self._ons_idx = _build_ons_index(self.orth_set)
        print(f"       ONS 母集団: {len(self.orth_set)} 語（unique 見出し本体）")

        # ── PNS（音韻近傍サイズ）──
        print("[5/5] PNS インデックス構築...")
        # 重複排除: 同一読み（モーラ列）は1回だけカウント
        self._orth2mora = {}
        phon_set = set()
        for y in df['読み'].dropna().unique():
            yc = primary_yomi(y)
            mt = tokenize_mora(yc)
            if mt:
                phon_set.add(mt)
        for _, row in df.drop_duplicates(subset='見出し本体').iterrows():
            m = row['見出し本体']
            y = primary_yomi(str(row['読み']))
            mt = tokenize_mora(y)
            if mt:
                self._orth2mora[m] = mt
        self._pns_idx = _build_pns_index(phon_set)
        print(f"       PNS 母集団: {len(phon_set)} 語（unique 読み）")

        print("       準備完了")


    def compute(self, word: str, reading: str) -> WordMetrics:
        """単一の (単語, 読み) ペアの心理言語学変数を計算する。"""
        yomi = normalize_reading(reading)
        mora = tokenize_mora(yomi)
        ons = _compute_ons(word, self._ons_idx) if word in self.orth_set else 0
        pns = _compute_pns(mora, self._pns_idx) if mora else 0
        old20 = _compute_old20(word, self.orth_list)
        return WordMetrics(
            word=word, reading_normalized=yomi,
            n_chars=len(word), n_morae=len(mora),
            ons=ons, pns=pns, old20=old20,
        )

    def compute_batch(self, pairs: List[Tuple[str, str]],
                      verbose: bool = True) -> pd.DataFrame:
        """複数の (単語, 読み) ペアを一括計算して DataFrame で返す。"""
        rows = []
        total = len(pairs)
        for i, (word, reading) in enumerate(pairs):
            if verbose and (i + 1) % 100 == 0:
                print(f"  {i+1}/{total}...")
            m = self.compute(word, reading)
            rows.append(m._asdict())
        return pd.DataFrame(rows)



# ╔═══════════════════════════════════════════════════════════╗
# ║  Demo                                                      ║
# ╚═══════════════════════════════════════════════════════════╝

if __name__ == '__main__':
    import os, time
    import urllib.request

    # ファイル名とダウンロード元の Raw URL を設定
    filename = "bunruidb.txt"
    url = "https://raw.githubusercontent.com/masayu-a/WLSP/master/bunruidb.txt"

    # カレントディレクトリにファイルが存在するかチェック
    if not os.path.exists(filename):
        print(f"'{filename}' が見つかりません。ダウンロードを開始します...")
        try:
            # ファイルをダウンロードして保存
            urllib.request.urlretrieve(url, filename)
            print("ダウンロードが正常に完了しました。")
        except urllib.error.URLError as e:
            print(f"ダウンロード中にネットワークエラーが発生しました: {e}")
        except Exception as e:
            print(f"予期せぬエラーが発生しました: {e}")
    else:
        print(f"'{filename}' は既にカレントディレクトリに存在します。ダウンロードをスキップします。")

    WLSP_PATH = filename

    engine = WLSP_metrics_Engine(WLSP_PATH, em_n_iter=20, em_max_chars=4)

    # ── 心理言語学変数の計算 ──
    test_pairs = [
        ('研究', 'ケンキュウ'),  ('国語', 'コクゴ'),
        ('会社', 'カイシャ'),    ('データ', 'データ'),
        ('コース', 'コース'),    ('独立行政法人', 'ドクリツギョウセイホウジン'),
        ('発表', 'ハッピョウ'),  ('熱心', 'ネッシン'),
        ('生活', 'セイカツ'),    ('先生', 'センセイ'),
    ]

    print("\n" + "=" * 80)
    print("心理言語学変数の計算結果")
    print("=" * 80)

    t0 = time.perf_counter()
    df = engine.compute_batch(test_pairs, verbose=False)
    t1 = time.perf_counter()
    print(f"\n計算時間: {t1-t0:.2f}s ({len(test_pairs)}語)")

    display_cols = ['word', 'reading_normalized', 'n_chars', 'n_morae',
                    'ons', 'pns', 'old20']
    print(df[display_cols].to_string(index=False))


'bunruidb.txt' が見つかりません。ダウンロードを開始します...
ダウンロードが正常に完了しました。
[1/5] WLSP 読み込み...
       101067 レコード
[2/5] レコード除外...
       101067 → 97630 レコード（3437 件除外）
[3/5] NFKC 正規化...
[4/5] ONS インデックス構築...
       ONS 母集団: 79316 語（unique 見出し本体）
[5/5] PNS インデックス構築...
       PNS 母集団: 67389 語（unique 読み）
       準備完了

心理言語学変数の計算結果

計算時間: 0.53s (10語)
  word reading_normalized  n_chars  n_morae  ons  pns  old20
    研究              けんきゅう        2        4   12   48   1.00
    国語                こくご        2        3  162   34   1.00
    会社               かいしゃ        2        3   64   47   1.00
   データ                でえた        3        3    2    8   1.85
   コース                こおす        3        3   20   26   1.00
独立行政法人      どくりつぎょうせいほうじん        6       12    0    0   4.00
    発表              はっぴょう        2        4  104   12   1.00
    熱心               ねっしん        2        4  137   15   1.00
    生活               せいかつ        2        4   75   25   1.00
    先生               せんせい        2        4  144   63   1.00
